In [7]:
import os
import time
import json

from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

import sys
import json
import subprocess
from pathlib import Path

In [ ]:
load_dotenv(find_dotenv())

# OpenAI API configuration
openai_key = os.getenv("OPENAI_API_KEY")

assert openai_key, "OPENAI_API_KEY not found in .env"

# Cloud model
openai_model = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=openai_key,
    temperature=0
)

# Local Ollama model
ollama_base_url = os.getenv(
    "OLLAMA_BASE_URL",
    "http://127.0.0.1:11434"
)

ollama_model_name = os.getenv(
    "OLLAMA_MODEL",
    "qwen2.5:3b"
)

ollama_model = ChatOpenAI(
    model=ollama_model_name,
    base_url=f"{ollama_base_url}/v1",
    api_key="ollama",
    temperature=0
)

print(f"Ollama model configured: {ollama_model_name}")
print(f"Ollama endpoint: {ollama_base_url}")

Ollama model configured: qwen2.5:3b
Ollama endpoint: http://127.0.0.1:11434


In [11]:
SERVER_PATH = Path("recon_server.py").resolve()

async def inspect_mcp_tools():
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(SERVER_PATH)]
    )

    async with stdio_client(
        server_params,
        errlog=subprocess.DEVNULL
    ) as (read, write):

        async with ClientSession(read, write) as session:
            await session.initialize()

            # Discover tools while the session is active
            tools_result = await session.list_tools()

            available_tools = [
                {
                    "name": tool.name,
                    "description": tool.description,
                    "input_schema": tool.inputSchema
                }
                for tool in tools_result.tools
            ]

            print(json.dumps(available_tools, indent=2))

            return available_tools

available_tools = await inspect_mcp_tools()

combined_findings = "\n\n".join(WEB_FINDINGS)

tool_descriptions = json.dumps(
    available_tools,
    indent=2
)

[
  {
    "name": "banner_grab",
    "description": "Connect to an allowlisted TCP target and read its initial service banner.",
    "input_schema": {
      "properties": {
        "target": {
          "title": "Target",
          "type": "string"
        },
        "port": {
          "title": "Port",
          "type": "integer"
        },
        "timeout": {
          "default": 2.0,
          "title": "Timeout",
          "type": "number"
        }
      },
      "required": [
        "target",
        "port"
      ],
      "title": "banner_grabArguments",
      "type": "object"
    }
  },
  {
    "name": "anonymous_ftp_check",
    "description": "Check whether an allowlisted FTP service permits anonymous login.",
    "input_schema": {
      "properties": {
        "target": {
          "title": "Target",
          "type": "string"
        },
        "port": {
          "default": 2121,
          "title": "Port",
          "type": "integer"
        },
        "timeout": {
        

NameError: name 'WEB_FINDINGS' is not defined

In [ ]:
selection_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an AI security analyst performing authorized,
read-only reconnaissance.

You have access to the following MCP tools:
{tools}

Rules:
1. Only recommend tools that appear in the provided tool list.
2. Only use the authorized target 127.0.0.1.
3. Web tools must use port 3000.
4. Service tools must use ports 2121 or 8080.
5. Do not recommend exploitation or destructive actions.
6. Select a tool only when additional investigation is justified.
7. If no additional investigation is justified, return NONE.
8. Do not execute tools.

Return JSON only, using this structure:

{{
    "tool_name": "tool name or NONE",
    "arguments": {{}},
    "reason": "Why this tool was selected"
}}
"""),
    ("user", """
Analyze these security findings and recommend the
most appropriate next investigation:

{findings}
""")
])

openai_selection_chain = (
    selection_prompt
    | openai_model
    | StrOutputParser()
)

ollama_selection_chain = (
    selection_prompt
    | ollama_model
    | StrOutputParser()
)

import time

# OpenAI recommendation
start_time = time.perf_counter()

openai_selection = openai_selection_chain.invoke({
    "tools": tool_descriptions,
    "findings": combined_findings
})

openai_time = time.perf_counter() - start_time


# Ollama recommendation
start_time = time.perf_counter()

ollama_selection = ollama_selection_chain.invoke({
    "tools": tool_descriptions,
    "findings": combined_findings
})

ollama_time = time.perf_counter() - start_time

print("OPENAI RECOMMENDATION")
print(openai_selection)
print(f"Time: {openai_time:.2f} seconds")

print("\nOLLAMA RECOMMENDATION")
print(ollama_selection)
print(f"Time: {ollama_time:.2f} seconds")

NameError: name 'combined_findings' is not defined

In [4]:
import os
import sys
import json
import time
import subprocess

from pathlib import Path
from dotenv import load_dotenv

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# --------------------------------------------------
# 1. Load environment variables
# --------------------------------------------------

load_dotenv()

openai_key = os.environ["OPENAI_API_KEY"]
openai_model_name = os.environ["OPENAI_MODEL"]

ollama_base_url = os.environ["OLLAMA_BASE_URL"]
ollama_model_name = os.environ["OLLAMA_MODEL"]


# --------------------------------------------------
# 2. Configure AI models
# --------------------------------------------------

openai_model = ChatOpenAI(
    model=openai_model_name,
    api_key=openai_key,
    temperature=0
)

ollama_model = ChatOpenAI(
    model=ollama_model_name,
    base_url=f"{ollama_base_url}/v1",
    api_key="ollama",
    temperature=0
)


# --------------------------------------------------
# 3. Locate MCP server
# --------------------------------------------------

SERVER_PATH = Path("recon_server.py").resolve()

if not SERVER_PATH.exists():
    raise FileNotFoundError(
        f"Could not find MCP server: {SERVER_PATH}"
    )


# --------------------------------------------------
# 4. Discover MCP tools and compare model selections
# --------------------------------------------------

def extract_text(result):
    return "\n".join(
        item.text
        for item in result.content
        if hasattr(item, "text")
    )

async def compare_model_selections():

    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(SERVER_PATH)]
    )

    async with stdio_client(
        server_params,
        errlog=subprocess.DEVNULL
    ) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            # Discover tools directly from MCP server
            tools_result = await session.list_tools()

            available_tools = [
                {
                    "name": tool.name,
                    "description": tool.description,
                    "input_schema": tool.inputSchema
                }
                for tool in tools_result.tools
            ]

            tool_descriptions = json.dumps(
                available_tools,
                indent=2
            )

            # Collect initial MCP reconnaissance results
            
            # Collect initial MCP reconnaissance results
            scanner_results = {}

            # Map discovered tools so we can verify availability
            tool_map = {
                tool.name: tool
                for tool in tools_result.tools
            }

            # Initial web and service reconnaissance
            scanner_tools = {
                # Web-side scanners
                "scan_http_headers": {
                    "target": "127.0.0.1",
                    "port": 3000
                },
                "enumerate_http_methods": {
                    "target": "127.0.0.1",
                    "port": 3000
                },
                "analyze_cookie_security": {
                    "target": "127.0.0.1",
                    "port": 3000
                },

                # Service-side scanners
                "banner_grab": {
                    "target": "127.0.0.1",
                    "port": 2121
                },
                "anonymous_ftp_check": {
                    "target": "127.0.0.1",
                    "port": 2121
                },
                "http_probe": {
                    "target": "127.0.0.1",
                    "port": 8080,
                    "path": "/"
                }
            }

            # Verify all six tools were discovered
            for tool_name in scanner_tools:
                if tool_name not in tool_map:
                    raise ValueError(
                        f"MCP tool not found: {tool_name}"
                    )

            # Require approval before running initial reconnaissance
            print("\nINITIAL RECONNAISSANCE")
            print("The following six scanners will execute:")

            for tool_name in scanner_tools:
                print(f"- {tool_name}")

            approval = input(
                "\nType APPROVE to execute these scanners: "
            )

            if approval != "APPROVE":
                raise RuntimeError(
                    "Initial reconnaissance cancelled."
                )

            # Execute all six scanners
            for tool_name, arguments in scanner_tools.items():

                print(f"\nRunning: {tool_name}")

                result = await session.call_tool(
                    tool_name,
                    arguments
                )

                scanner_results[tool_name] = extract_text(result)

                print(f"Completed: {tool_name}")

            # Combine live MCP results for AI analysis
            combined_findings = json.dumps(
                scanner_results,
                indent=2
            )

            print("\nINITIAL SCANNER RESULTS")
            print(combined_findings)                


            # --------------------------------------------------
            # 5. Create shared selection prompt
            # --------------------------------------------------

            selection_prompt = ChatPromptTemplate.from_messages([
                ("system", """
                    You are an AI security analyst performing authorized,
                    read-only reconnaissance.

                    Available MCP tools:
                    {tools}

                    Rules:
                    1. Only recommend tools in the provided tool list.
                    2. Only use the authorized target 127.0.0.1.
                    3. Web tools must use port 3000.
                    4. Service tools must use ports 2121 or 8080.
                    5. Do not recommend exploitation or destructive actions.
                    6. Select a tool only when additional investigation is justified.
                    7. If no additional investigation is justified, select NONE.
                    8. Do not execute tools.

                    Return JSON only:

                    {{
                        "tool_name": "tool name or NONE",
                        "arguments": {{}},
                        "reason": "Why this tool was selected"
                    }}
                    """),
                                    ("user", """
                    Analyze these security findings and recommend
                    the most appropriate next investigation:

                    {findings}
                    """)
            ])


            # --------------------------------------------------
            # 6. Create separate model chains
            # --------------------------------------------------

            openai_selection_chain = (
                selection_prompt
                | openai_model
                | StrOutputParser()
            )

            ollama_selection_chain = (
                selection_prompt
                | ollama_model
                | StrOutputParser()
            )


            # --------------------------------------------------
            # 7. Get OpenAI recommendation
            # --------------------------------------------------

            start_time = time.perf_counter()

            openai_selection = openai_selection_chain.invoke({
                "tools": tool_descriptions,
                "findings": combined_findings
            })

            openai_time = time.perf_counter() - start_time


            # --------------------------------------------------
            # 8. Get Ollama recommendation
            # --------------------------------------------------

            start_time = time.perf_counter()

            ollama_selection = ollama_selection_chain.invoke({
                "tools": tool_descriptions,
                "findings": combined_findings
            })

            ollama_time = time.perf_counter() - start_time


            # --------------------------------------------------
            # 9. Display comparison
            # --------------------------------------------------

            print("\n" + "=" * 50)
            print("OPENAI RECOMMENDATION")
            print("=" * 50)

            print(openai_selection)
            print(f"\nExecution time: {openai_time:.2f} seconds")


            print("\n" + "=" * 50)
            print("OLLAMA RECOMMENDATION")
            print("=" * 50)

            print(ollama_selection)
            print(f"\nExecution time: {ollama_time:.2f} seconds")


            return {
                "available_tools": available_tools,
                "openai_selection": openai_selection,
                "ollama_selection": ollama_selection,
                "openai_time": openai_time,
                "ollama_time": ollama_time
            }


# --------------------------------------------------
# 10. Run comparison
# --------------------------------------------------

comparison_results = await compare_model_selections()


INITIAL RECONNAISSANCE
The following six scanners will execute:
- scan_http_headers
- enumerate_http_methods
- analyze_cookie_security
- banner_grab
- anonymous_ftp_check
- http_probe

Running: scan_http_headers
Completed: scan_http_headers

Running: enumerate_http_methods
Completed: enumerate_http_methods

Running: analyze_cookie_security
Completed: analyze_cookie_security

Running: banner_grab
Completed: banner_grab

Running: anonymous_ftp_check
Completed: anonymous_ftp_check

Running: http_probe
Completed: http_probe

INITIAL SCANNER RESULTS
{
  "scan_http_headers": "{\n  \"scanner\": \"HTTP Security Header Scanner\",\n  \"target\": \"http://127.0.0.1:3000/\",\n  \"http_status\": 200,\n  \"findings\": [\n    {\n      \"header\": \"Content-Security-Policy\",\n      \"status\": \"MISSING\",\n      \"observed_value\": null,\n      \"finding\": \"Content-Security-Policy was not present in the response.\"\n    },\n    {\n      \"header\": \"X-Frame-Options\",\n      \"status\": \"PRESEN